# CARLA-GeAR 2D Detection

In [ ]:
!git clone https://github.com/retis-ai/CARLA-GeAR.git
!git clone https://github.com/retis-ai/PatchAttackTool.git

In [ ]:
!pip install opencv-python PyYAML matplotlib
!pip install torch torchvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/retis-ai/PatchAttackTool.git
%cd PatchAttackTool
!pip install -r requirements.txt
!pip install torch torchvision opencv-python matplotlib pycocotools tqdm pandas


In [ ]:
!cp configs/examples/frcnn.yml configs/ptod/my_config.yml

In [ ]:
!pip install kornia

In [ ]:
!rm -rf PatchAttackTool
!git clone https://github.com/retis-ai/PatchAttackTool.git

In [ ]:
# This patch updates eval_script.py to fix the yaml.load error
!sed -i "s/yaml.load(fp)/yaml.load(fp, Loader=yaml.SafeLoader)/" eval_script.py


In [ ]:
import sys
sys.path.append("/content/PatchAttackTool")

In [ ]:
cfg_dict = {
    "task": "od",
    "model": {
        "name": "fasterrcnn_resnet50_fpn",
        "pretrained": True
    },
    "data": {
        "dataset": {
            "name": "coco",
            "root": "/content/drive/MyDrive/billboard01",
            "train_split": "train",
            "val_split": "val",
            "ann_path_train": " /content/drive/MyDrive/billboard01/annotations/train/coco_annotations.json",
            "ann_path_val": "/content/drive/MyDrive/billboard01/annotations/val/coco_annotations.json",
            "use_crowd": False,
            "return_masks": False
        }
    },
    "attack": {
        "apply_patch": False,
        "patch_size": 64
    },
    "device": {
        "use_cuda": False,
        "gpu": 0
    },
    "seed": 42
}

os.makedirs("PatchAttackTool/configs/ptod", exist_ok=True)
config_path = "PatchAttackTool/configs/ptod/my_config.yml"
with open(config_path, "w") as f:
    yaml.dump(cfg_dict, f)


In [ ]:
# Force skip CUDA call if use_cuda=False
!sed -i 's/torch\.cuda\.set_device(cfg\["device"\]\["gpu"\])/if cfg["device"]["use_cuda"]: torch.cuda.set_device(cfg["device"]["gpu"])/' eval_script.py
!sed -i 's/torch\.cuda\.set_device(cfg\["device"\]\["gpu"\])/if cfg["device"]["use_cuda"]: torch.cuda.set_device(cfg["device"]["gpu"])/' optimization_script.py


In [ ]:
from task_interfaces import get_task_interface

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

task = cfg["task"]
interface = get_task_interface(task, cfg)


In [ ]:
!ls /content/drive/MyDrive/2dod/2dod/billboard01/images


In [ ]:
import torch
from torchvision.datasets import CocoDetection
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torch.utils.data import DataLoader
import torchvision.transforms as T

# Custom COCO wrapper
class CocoDetectionFormatted(CocoDetection):
    def __getitem__(self, index):
        img, target = super().__getitem__(index)
        boxes = []
        labels = []

        for obj in target:
            bbox = obj["bbox"]  # COCO: [x, y, w, h]
            x_min, y_min, width, height = bbox
            x_max = x_min + width
            y_max = y_min + height
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(obj["category_id"])

        target_dict = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }

        return img, target_dict

root = "/content/drive/MyDrive/2dod/2dod/billboard01/images/train"
ann_train = "/content/drive/MyDrive/2dod/2dod/billboard01/annotations/train/coco_annotations.json"

# --- Transform ---
transform = T.Compose([T.ToTensor()])

# --- Dataset and Loader ---
train_dataset = CocoDetectionFormatted(root=root, annFile=ann_train, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

# --- Model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = fasterrcnn_resnet50_fpn(pretrained=True)
model.to(device)

# --- Optimizer ---
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# --- Training Loop (1 epoch for demo) ---
model.train()
for images, targets in train_loader:
    images = [img.to(device) for img in images]
    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

    loss_dict = model(images, targets)
    losses = sum(loss for loss in loss_dict.values())

    optimizer.zero_grad()
    losses.backward()
    optimizer.step()

    print(f" Loss: {losses.item():.4f}")

